# Item 99: Consider `memoryview` and `bytearray` for Zero-Copy

Interactions with `bytes`

## Notes

-   Python requires extra effort to parallelise CPU-bound computation
    (See [Item 79](../../Chapter_09/Item_079/item_079.qmd) and [Item
    94](../Item_094/item_094.qmd))
-   But, can support high-throughput parallel I/O (See [Item
    68](../../Chapter_09/Item_068/item_068.qmd) and [Item
    75](../../Chapter_09/Item_075/item_075.qmd))
-   However, understanding the tools available and how to use them
    *without* leading to slow code can require some skill
-   For example, consider a media-streaming server
    -   Users don’t need to download a video in advance
    -   Users can move forward or backward within a video
-   We might have functions to implement this by converting a time-code
    to a index and returning the associated chunk of data

In [1]:
import os # for demo only

def timecode_to_index(video_id, timecode):
    # Returns byte offser in the video data
    return 0  # placeholder


def request_chunk(video_id, byte_offset, size):
    # Returns size bytes of video_id's data from the offset
    # simulate by returning random data
    return os.urandom(size)


video_id = 1
timecode = "01:09:14:28"
byte_offset = timecode_to_index(video_id, timecode)
size = (8**2)
video_data = request_chunk(video_id, byte_offset, size)

print(f"{video_id=}, {timecode=}, {byte_offset=}, {video_data=}")

video_id=1, timecode='01:09:14:28', byte_offset=0, video_data=b'\xb0\xac2\xaa\xcfD\xe1\xb2\xb4k8Z}\x8e\xa0\xb7P\xa7\n#\xf22^\xe0\xbc\xc3\xaf(?T\xedn\xc9m\xd5K\xd4\xd8\xe9\xe2\xc2\x04T\xbd.\xf74\xb3\xdb\xc8\x19\xd3q\xc4\x96\xd8\x9c\xe4|\xc8\xa2p\\5'

-   How do we now implement the server-side handler that receives
    `request_chunk`
    -   Must then return the associated video data chunk
-   First we assume that the program is driven by an `asyncio` process
    (See [Item 76](../../Chapter_09/Item_076/item_076.qmd))
    -   Now want to focus on how to handle extracting the chunk
    -   Assume video is cached memory
    -   Extracted then sent over a socket back to a client

In [2]:
import os # for demo only

def timecode_to_index(video_id, timecode):
    # Returns byte offser in the video data
    return 0  # placeholder


def request_chunk(video_id, byte_offset, size):
    # Returns size bytes of video_id's data from the offset
    return video_data[byte_offset : byte_offset + size]

# Adding in the handling

# simulate a socket connection
class NullSocket:
    def __init__(self):
        self.handle = open(os.devnull, "wb")

    def send(self, data):
        self.handle.write(data)

socket = NullSocket() # represents client socket connection
size = (8 ** 2) # Requested chunk size
video_data = os.urandom(20 * size) # Bytes containing data for video_id

video_id = 1
timecode = "01:09:14:28"

byte_offset = timecode_to_index(video_id, timecode)
chunk = request_chunk(video_id, byte_offset, size)
socket.send(chunk)

print(f"Sent {chunk=} over socket")

Sent chunk=b'\x95/\xc4\x0b\x95s\xcf\xdfl@\x96\xc4\xe1T\xcddb^\xe1w\xf2EB\xa5\xe1\x9ez\xeaJ\xfa\xc3F=)\x82l\x95$\x162\x01yl~06&\x8b\x96\xf0\xe5^W\n\xb3k\xfb\x00\xfdM\x0f\xa2\x7f\x8f' over socket

-   Latency and throughput determined by two factors
    1.  How long to slice (See [Item
        14](../../Chapter_02/Item_014/item_014.qmd)) the chunk from
        `video_data`
    2.  How long to transmit over a socket
-   Focusing just on point 1, we can microbenchmark how long fetching a
    chunk takes.
    -   We’ll also exclude the function call wrapper
    -   Here we’ll set the size to $20$ MB.

In [3]:
import timeit

size = 20 * (1024**2)
video_data = os.urandom(20 * size) # Bytes containing data for
byte_offset = 0

def run_test():
    chunk = video_data[byte_offset : byte_offset + size]

result = ( timeit.timeit(stmt="run_test()", globals=globals(), number=100) / 100 )

print(f"{result:0.9f} seconds")

0.000964729 seconds

-   This takes about $5$ milliseconds
-   Theoretical server maximum throughput is thus, limited by video
    extraction speed as

$$
\begin{align}
    \frac{20 \text{ MB}}{5 \text{ ms}} &= 4 \text{ GB}\text{s}^{-1}
\end{align}
$$

-   Server also limited to,

$$
\begin{align}
    \frac{1 \text{ CPU=second}}{5 \text{ ms}} &= 200 \text{ clients in parallel}
\end{align}
$$

-   But we already know that `asyncio` should be able to scale up to
    tens of thousands of simultaneous connections
-   The slowdown is because as discussed slices create copies
    -   Copying consumes CPU time
-   Instead we can use `memoryview`
    -   A built-in type for handling the CPython `buffer` protocol
        -   Low-level C API allowing Python runtime and C extensions
            (See [Item 96](../Item_096/item_096.qmd)) to access
            underlying data buffers
            -   Can then treat them as `bytes` instances
        -   Since Python 3.12 the buffer protocol is also emulatable in
            python
-   `memoryview` can be sliced to create a new `memoryview` without a
    copy

In [4]:
data = b"shave and a haircut, two bits"
view = memoryview(data)
chunk = view[12:19]

print(chunk)
print("Size:            ", chunk.nbytes)
print("Data in view:    ", chunk.tobytes())
print("Underlying data: ", chunk.obj)

Size:             7
Data in view:     b'haircut'
Underlying data:  b'shave and a haircut, two bits'

-   These *zero-copy* operations can significantly speed-up code that
    heavily processes memory, e.g.
    1.  I/O-bound access
    2.  Heavy numerical mathematics (e.g. Numpy)
-   Using `memoryview` as a drop-in replacement for our video serving
    service

In [5]:
import timeit

size = 20 * (1024**2)
video_data = os.urandom(20 * size) # Bytes containing data for
video_view = memoryview(video_data)
byte_offset = 0

def run_test():
    chunk = video_view[byte_offset : byte_offset + size]

result = ( timeit.timeit(stmt="run_test()", globals=globals(), number=100) / 100 )

print(f"{result:0.9f} seconds")

0.000000199 seconds

-   This should run in a several hundred nanoseconds
-   So an order of magnitude faster than the `bytes` slicing technique
-   Our new theoretical maximum throughput is then

$$
\begin{align}
\frac{20 \text{ MB}}{250 \text{ ns}} &= 80 \text{ TB}\text{s}^{-1}
\end{align}
$$

-   Or in terms of parallel clients

$$
\begin{align}
\frac{1 \text{ CPU-second}}{250 \text{ ns}} &= 4 \times 10^{9}
\end{align}
$$

-   So four million clients. Now the program should be bound by the
    socket performance rather than CPU constraints.

-   Now consider a reversed process

    -   Users must submit live video streams that are then broadcast out
        to viewers

-   We need to store incoming video data

    -   Cache it for clients to read from

In [6]:
import os

def timecode_to_index(video_id, timecode):
    # Returns byte offser in the video data
    return 0  # placeholder

# socket connection from client


size = (4 ** 2) # Incoming chunk size
video_data = os.urandom(20 * size) # Bytes containing data for video
video_cache = video_data[:]

video_id = 1
timecode = "01:09:14:28"
byte_offset = timecode_to_index(video_id, timecode) # Incoming buffer position
video_view = memoryview(video_cache)


class MockIncomingSocket:

    def recv(self, size):
        return video_view[byte_offset : byte_offset + size]

    def recv_into(self, buffer):
        source_data = video_view[byte_offset : byte_offset + size]
        buffer[:] = source_data

socket = MockIncomingSocket()
chunk = socket.recv(size)
before = video_view[:byte_offset]
after = video_view[byte_offset + size:]

new_cache = b"".join([before, chunk, after])

print(f"Updated the cache: {new_cache=}")

Updated the cache: new_cache=b'\x88\xf0-\x013\x99\x105\x84\x02j\xd0\xd1\xb9:\xdf\xa7XR\x93\x7f\x95#\xff\xc3x\xa5i8\xbf \xc1p\xfe\xc9\xb4\x94f\x1dd_\xac\xa9\x9aw\x1c:\x98\xc1\xfeB\x18\x1f\xefF\xedp\xbdmx\xd8(\x8d\xb2L\x17Y\xbfu1\xd4\xaau\xf4\x1f\x94\x82\x91\x07\xc2]\xea\x13\x81^\xb1\xc9I?3\x1d\xc4G\xee\x87^\x94\x89U\r\xef\xc9\x06\xae\x92\xa9\xc5\xd4\xbd`\xc1\xd6\xac\xec\x9c\x00\x11\xfaP\xbaS\xc9d9\xf7\xb6<\x0b\x84x\nAC\x7fg\x13\xa3\x82\x10#)\xdb\xdd\x02\xd4\xf3{\x1e\xdf\x01\xb2\xd7u#\xe2\x05u\xa5\x02\xbe\x1e\x81|\xf6\x1db\xedG=S\xe7\x7f\x03\xc0\x17\x8e\xeb@\x86\xa209\xd4\xcb\x97\x9e\xd6\x0c\xeb\xefx\x8f\xd3!\x00\xa4\x17>Q\xf1\xe8\x1a\xcf\xbfx\xb4S\x02>7\xf5u3{\xed\x07\xdan\xa8\x95\x90W\xa5k\xb5v\xef\xfe\x17\xe2*\x9f\x1cC%o\x95\xbawz}\x10\x02\xef\xb9ke\x87\x8c\x98\xcdw\xb33\xce\xdbT\xfe\xa9fiJn\x1a\x1c\x17Y8#\xa1M\xe5\x1b\x1d\x01\x80\x87\x89pC\xc4\x02\x979.Z\xe0\x95<\xc2\\+\xc2\x87O\x9c\xfa(\x84\x11_9Fq\xb3tQ@\x86Y\x13G\x85\xf2\xd3eJ\x1dHe'

-   `socket.recv` returns a `bytes` instance
    -   Splice this into the existing cache
    -   Insert at the current `byte_offset` via slicing and `bytes.join`
-   Now need to profile the timing

In [7]:
import timeit
import os

class MockIncomingSocket:

    def recv(self, size):
        return video_view[byte_offset : byte_offset + size]

    def recv_into(self, buffer):
        source_data = video_view[byte_offset : byte_offset + size]
        buffer[:] = source_data

socket = MockIncomingSocket()
size = (1024 ** 2) # Incoming chunk size
video_data = os.urandom(20 * size) # Bytes containing data for video
video_cache = video_data[:]
video_view = memoryview(video_cache)
byte_offset = 1234 # pick arbitrary point in the middle

def run_test():
    chunk = socket.recv(size)
    before = video_view[:byte_offset]
    after = video_view[byte_offset + size : ]
    new_cache = b"".join([before, chunk, after])

result = (timeit.timeit(stmt="run_test()", globals=globals(), number=100,) / 100)

print(f"{result:0.9f} seconds")

0.000921429 seconds

-   This takes about three milliseconds to receive $1$ MB and update the
    cache.
-   Maximum throughput to receive is then

$$
\begin{align}
\frac{1 \text{ MB}}{ 3 \text{ ms}} &\approx 330 \text{ MB}\text{s}^{-1}
\end{align}
$$

-   Means we are limited to about $300$ simultaneously streaming clients
-   Can use `bytearray` instead of `memoryview`
    -   `bytes` are immutable like strings

In [8]:
some_bytes = b"hello"
some_bytes[0] = 0x79

-   `bytearray` is effectively a mutable version of `bytes`
    -   Can overwrite indices
-   `bytearray` values are integers rather than bytes

In [9]:
array = bytearray(b"hello")
array[0] = 0x79
print(array)

bytearray(b'yello')

-   Can still wrap a `bytearray` in a `memoryview` to avoid extra copies
    -   Then can slice the `memoryview` and modify to overwrite the
        underlying `bytearray`

In [10]:
array = bytearray(b"row, row, row your boat")
view = memoryview(array)
write_view = view[3:13]
write_view[:] = b"-10 bytes-"
print(array)

bytearray(b'row-10 bytes- your boat')

-   Library methods in Python user the buffer protocol for fast data
    receipt or reading, e.g.
    1.  `socket.recv_into`
    2.  `RawIOBase.read_into`
-   These methods avoid creating copies and allocating memory
    -   Received data goes into existing buffer
-   We can convert our program to use `recv_into` and a `memoryview`
    slice to speed up our broadcasting method

In [11]:
class MockIncomingSocket:

    def recv(self, size):
        return video_view[byte_offset : byte_offset + size]

    def recv_into(self, buffer):
        source_data = video_view[byte_offset : byte_offset + size]
        buffer[:] = source_data

# socket connection from client
socket = MockIncomingSocket()

size = (4 ** 2) # Incoming chunk size
byte_offset = 1234
video_data = os.urandom(20 * size) # Bytes containing data for video
video_cache = video_data[:]
video_view = memoryview(video_cache)

video_array = bytearray(video_cache)
write_view = memoryview(video_array)

chunk = write_view[byte_offset : byte_offset + size]
socket.recv_into(chunk)

print(f"New cache: {video_cache=}")

New cache: video_cache=b'u\xc78N\xd9_\xcc\xcc\x81F\\v\x91krD\x0b\xe0\xf0\x83+\x0cXWbI]\xdf\xb0!\xa0\x9f\xd2o\xfax]\x05\x9bj\x03<\xe3\x97\xee\xcd\xbdO\x9a3\xed\x95de\xbec0\x04\xe1d\xb8\xda\x90\n\xff\x9d\xa7\x1b\xf6\xd9\n\xf7se\xca\xba&@\xaa\x7f\x18\x0b\xfd\x1a\xe5\xe21EL0vO\xd6G\xf4F\xabI&\x8f\xb1\x8c\x0c1\x0frO\xf5\x9e\rp\x87Uj\xea\xda\xd2Zd\xbc\xd1\x91\xe2\x99\x9d\tn\xdf^3\xdc\xf1\xb0u\x1bMY\x9a\xe2R\xa2\x83\x93_\xbcn\x02\xa2\xc7\xff\x82\x8b\xabVz\xe5V\xf6\x92`\xa2\xac\x14\xdb\xba\xfb}%\x94\xaa\xb1\xa3%8\xfc-\xd7\xb9\xfe\xe4\x15|\xa3a\x16aq_7\xa5\x05Gd1i\xa6\xf7\xadk\xc8\x7f\xdb\xcf\xd0\x9d7\x8b\x04\xb1\xd9\xdc\xa3\xd6\xb9\x99\xfd?\xb9>\xe4 _\xc02\xa8\xe6\x9c\x8f\xc2\x15\xa7F\xf1Eo;Z6?\xbd\xb4\x12\x7f\x19\xd0\xf8H2\xf6VU\xd4\xe0\x987\x87\xc4\xb5\x03\xd6r\xec\xc1M2\xd7\xc0*\xc56~\xa1\xb0\xf8\x03\x9a\xaa\x02\xd4;\n\xce:4\xf1YC\xd7t8\xad[\xe1{\xe7,w\xfd\x89\xe2Ku\x8d\xc6\xd5F\x0f\xf5\x9bq`$T\x86!\x07\xeb\xa5\xc2J'

-   We can again microbenchmark the result for a $1$ MB chunk

In [12]:
import timeit
import os

class MockIncomingSocket:

    def recv(self, size):
        return video_view[byte_offset : byte_offset + size]

    def recv_into(self, buffer):
        source_data = video_view[byte_offset : byte_offset + size]
        buffer[:] = source_data

# socket connection from client
socket = MockIncomingSocket()

size = (1024 ** 2) # Incoming chunk size
byte_offset = 1234
video_data = os.urandom(20 * size) # Bytes containing data for video
video_cache = video_data[:]
video_view = memoryview(video_cache)

video_array = bytearray(video_cache)
write_view = memoryview(video_array)


def run_test():
    chunk = write_view[byte_offset : byte_offset + size]
    socket.recv_into(chunk)

result = (
    timeit.timeit(stmt="run_test()", globals=globals(), number=100) / 100
)

print(f"{result:0.9f} seconds")

0.000033280 seconds

-   On my machine this takes about $90 \;\mu\text{s}$. Which means we
    could support,

$$
\begin{align}
    \frac{1 \text{ MB}}{90 \; \mu\text{s}} &= 11 \text{ GB}\text{s}^{-1}
\end{align}
$$

-   Which also supports,

$$
\begin{align}
    \frac{11 \text{ GB}}{1 \text{MB}} &= 11,000 \text{ processes}
\end{align}
$$

-   Much better scalability

## Things to Remember

-   `memoryview` provides zero-copy methods for reading and writing to
    slices of objects supporting the buffer protocol
-   `bytearray` built-in provides a mutable `bytes`-like type
    -   Can be used for zero-copy data reads
    -   Works with functions like `socket.recv_into`
-   `memoryview` can wrap a `bytearray`
    -   Let’s received data to be spliced into an existing buffer
    -   No need for extra copies